can by torch or hf opt

# torch to onnx by torch
 - image size
 - ckpt

In [ ]:
!pip install --upgrade --upgrade-strategy eager optimum[onnxruntime-gpu]
!pip install --upgrade huggingface_hub

In [ ]:
import torch
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor
CHECKPOINT = "/home/jupyter/test/jetson/checkpoint-552"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RTDetrForObjectDetection.from_pretrained(CHECKPOINT, use_safetensors=True).to(DEVICE)
processor = RTDetrImageProcessor.from_pretrained(CHECKPOINT)
model.eval()
dummy_input = torch.randn(1, 3, 960, 960).to(DEVICE)
torch.onnx.export(
  model,
  dummy_input,
  "test3.onnx",
  export_params=True,
  opset_version=17,
  input_names=["input"],
  output_names=['labels', 'boxes', 'scores']
)

onnx 32 to 16/mix

In [ ]:
!pip install onnx onnxconverter-common

In [ ]:
import onnx
from onnxconverter_common import float16

model = onnx.load("test3.onnx")
model_fp16 = float16.convert_float_to_float16(model)
onnx.save(model_fp16, "test3_fp16.onnx")


/home/jupyter/envhf/lib/python3.10/site-packages/onnxconverter_common/float16.py:43: UserWarning: the float32 number 2.6624670822171524e-44 will be truncated to 1e-07
  warnings.warn("the float32 number {} will be truncated to {}".format(pos_min, min_positive_val))
/home/jupyter/envhf/lib/python3.10/site-packages/onnxconverter_common/float16.py:53: UserWarning: the float32 number -2.802596928649634e-45 will be truncated to -1e-07
  warnings.warn("the float32 number {} will be truncated to {}".format(neg_max, -min_positive_val))
/home/jupyter/envhf/lib/python3.10/site-packages/onnxconverter_common/float16.py:43: UserWarning: the float32 number 6.305843089461677e-44 will be truncated to 1e-07
  warnings.warn("the float32 number {} will be truncated to {}".format(pos_min, min_positive_val))
/home/jupyter/envhf/lib/python3.10/site-packages/onnxconverter_common/float16.py:53: UserWarning: the float32 number -4.203895392974451e-44 will be truncated to -1e-07
  warnings.warn("the float32 numb

# inference

 - onnx_model_path
 - label
 - image size
 - image path
 - float type 32/16

In [8]:
import numpy as np
from PIL import Image
import onnxruntime as ort

from typing import List, Tuple

# Load the ONNX model with the specified providers
onnx_model_path = "/home/jupyter/test/jetson/hf_onnx/model.onnx"
providers = ['CUDAExecutionProvider', 'OpenVINOExecutionProvider', 'CPUExecutionProvider']
ort_session = ort.InferenceSession(onnx_model_path, providers=providers)

# Define a function to preprocess the image
def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    original_size = list(image.size)
    image = image.resize((960, 960))  # Resize to model input size
    image = np.array(image).astype(np.float32) / 255.0  # Normalize ## np.float16
    image = np.transpose(image, (2, 0, 1))  # Change to CHW format
    return original_size, np.expand_dims(image, axis=0)  # Add batch dimension

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def box_convert(boxes: np.ndarray, in_fmt: str = 'cxcywh', out_fmt: str = 'xyxy') -> np.ndarray:
    """
    Convert bounding boxes from one format to another.

    Parameters:
        boxes (np.ndarray): An array of shape (M, N, 4) representing bounding boxes.
                            Each box is defined by four values: (center_x, center_y, width, height).
        in_fmt (str): The input format of the bounding boxes. Default is 'cxcywh'.
        out_fmt (str): The desired output format of the bounding boxes. Default is 'xyxy'.

    Returns:
        np.ndarray: An array of shape (M, N, 4) representing bounding boxes in the 'xyxy' format,
                    defined by (x1, y1, x2, y2).
    """
    
    if in_fmt == 'cxcywh' and out_fmt == 'xyxy':
        # Ensure boxes have the shape (M, N, 4)
        if boxes.ndim != 3 or boxes.shape[-1] != 4:
            raise ValueError("Input boxes must have shape (batch_size, query, 4)")

        # Extracting the center coordinates and dimensions
        cx, cy, w, h = boxes[..., 0], boxes[..., 1], boxes[..., 2], boxes[..., 3]
        
        # Calculate the top-left and bottom-right coordinates
        x1 = cx - 0.5 * w  # Top-left x-coordinate
        y1 = cy - 0.5 * h  # Top-left y-coordinate
        x2 = cx + 0.5 * w  # Bottom-right x-coordinate
        y2 = cy + 0.5 * h  # Bottom-right y-coordinate
        
        # Stack the results to form the output format (xyxy)
        return np.stack([x1, y1, x2, y2], axis=-1)
    else:
        raise ValueError("Unsupported format conversion")


def forward(outputs: List[np.ndarray], orig_target_sizes: np.ndarray, score_threshold: float = 0.5) -> List[dict]:
    """
    Provided function with post-processing to output scores, labels, and bounding boxes.
    
    :param outputs: List of model outputs (e.g., logits and boxes).
    :param orig_target_sizes: np.ndarray representing original image sizes.
    :param score_threshold: Minimum score threshold for filtering bounding boxes (default: 0.5).
    
    :return: List of dictionaries containing labels, boxes, and scores.
    """
    logits, boxes = outputs[0], outputs[1]

    # Convert boxes from 'cxcywh' format to 'xyxy' format
    # Multiply the bounding boxes by the original target sizes
    bbox_pred = box_convert(boxes, in_fmt='cxcywh', out_fmt='xyxy')
    bbox_pred *= np.expand_dims(orig_target_sizes.repeat(2, axis=0).reshape(1, -1), axis=1)

    # Calculate scores
    scores = sigmoid(logits)
    scores = scores.reshape((1, -1))

    # Find the indices of the top 300 scores along the last axis
    indices = np.argpartition(scores.reshape((1, -1)), -300, axis=-1)[..., -300:]

    # Sort the indices to get the final indices and scores
    sorted_indices = np.take_along_axis(indices, np.argsort(np.take_along_axis(scores, indices, axis=-1), axis=-1), axis=-1)
    # sorted_indices = sorted_indices[..., ::-1]

    # Get the corresponding scores
    sorted_scores = np.take_along_axis(scores, sorted_indices, axis=-1)
    labels = sorted_indices % 2
    sorted_indices = sorted_indices // 2

    # Expand the index to match the shape of bbox_pred
    index_expanded = np.expand_dims(sorted_indices, axis=-1)  # Shape: (batch_size, top_k, 1)

    # Repeat the index to match the last dimension of bbox_pred
    index_repeated = np.repeat(index_expanded, bbox_pred.shape[-1], axis=-1)  # Shape: (batch_size, top_k, bbox_dimensions)

    # Gather bounding boxes based on the expanded and repeated indices
    boxes = np.take_along_axis(bbox_pred, index_repeated, axis=1)  # Shape: (batch_size, top_k, bbox_dimensions)

    # Apply score threshold
    mask = sorted_scores >= score_threshold
    labels = [labels[mask]]
    boxes = [boxes[mask]]
    scores = [sorted_scores[mask]]

    # Create results dictionary
    results = []
    for lab, box, sco in zip(labels, boxes, scores):
        result = dict(labels=lab, boxes=box, scores=sco)
        results.append(result)

    return results

def inference(image: str, score_threshold: float = 0.3) -> List[dict]:
    """
    Main inference function to get labels, scores, and bounding boxes from an input image.

    Parameters:
        image (str): Path to the input image file.
        score_threshold (float): Minimum score threshold for filtering predictions. Default is 0.5.

    Returns:
        results (list): A list containing the detected labels, scores, and bounding boxes.
    """
    original_size, inputs = preprocess_image(image)
    outputs = ort_session.run(None, {ort_session.get_inputs()[0].name: inputs})

    model_outputs = outputs
    orig_image_sizes = np.array([original_size])  # Replace with actual image sizes
    results = forward(model_outputs, orig_image_sizes, score_threshold=score_threshold)
    return results

# example
# Perform inference on the test dataset
url = '/home/jupyter/test/jetson/4.jpg'
inference(url)

/home/jupyter/envhf/lib/python3.10/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:118: UserWarning: Specified provider 'OpenVINOExecutionProvider' is not in available provider names.Available providers: 'TensorrtExecutionProvider, CUDAExecutionProvider, CPUExecutionProvider'
  warnings.warn(
2025-05-02 05:12:32.799296110 [W:onnxruntime:, transformer_memcpy.cc:83 ApplyImpl] 6 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2025-05-02 05:12:33.334528176 [W:onnxruntime:Default, scatter_nd.h:51 ScatterNDWithAtomicReduction] ScatterND with reduction=='none' only guarantees to be correct if indices are not duplicated.
2025-05-02 05:12:33.334568202 [W:onnxruntime:Default, scatter_nd.h:51 ScatterNDWithAtomicReduction] ScatterND with reduction=='none' only guarantees to be correct if 

[{'labels': array([0, 0]),
  'boxes': array([[1046.6968 ,  478.3191 , 1533.4634 ,  765.9712 ],
         [ 400.1251 ,  850.7314 ,  443.18683,  876.2796 ]], dtype=float32),
  'scores': array([0.36762163, 0.3954631 ], dtype=float32)}]

# validation 

In [9]:
import torch

from PIL import Image
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor

image_path = "/home/jupyter/test/jetson/4.jpg"
image = Image.open(image_path)

image_processor = RTDetrImageProcessor.from_pretrained("/home/jupyter/test/jetson/checkpoint-552")
model = RTDetrForObjectDetection.from_pretrained("/home/jupyter/test/jetson/checkpoint-552")

inputs = image_processor(images=image, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

results = image_processor.post_process_object_detection(outputs, target_sizes=torch.tensor([image.size[::-1]]), threshold=0.3)

for result in results:
    for score, label_id, box in zip(result["scores"], result["labels"], result["boxes"]):
        score, label = score.item(), label_id.item()
        box = [round(i, 2) for i in box.tolist()]
        #print(f"{model.config.id2label[label]}: {score:.2f} {box}")
results


[{'scores': tensor([0.3625, 0.3377, 0.3145]),
  'labels': tensor([0, 0, 0]),
  'boxes': tensor([[ 400.0156,  850.7539,  443.1992,  876.2530],
          [1044.0638,  476.3843, 1533.4213,  763.8090],
          [1030.6672,  288.1363, 1533.3781,  770.2939]])}]

In [ ]:
%pip install --upgrade transformers

# torch to onnx by hf optimum

In [ ]:
!pip install --upgrade --upgrade-strategy eager optimum[onnxruntime-gpu]

In [ ]:
!pip install --upgrade huggingface_hub

In [ ]:
!pip install git+https://github.com/huggingface/optimum.git

fine-tune

In [1]:
from transformers import AutoModelForObjectDetection

# Load your model
model_name = "/home/jupyter/test/jetson/checkpoint-552"
redetr_model = AutoModelForObjectDetection.from_pretrained(model_name)
# redetr_model.save_pretrained("quantized_model/redetr", save_config=True, safe_serialization=False)

/home/jupyter/envhf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# could make use of PR https://github.com/huggingface/optimum/pull/1930 to run the following
!optimum-cli export onnx -m /home/jupyter/test/jetson/checkpoint-552 --task object-detection --framework 'pt' hf_onnx  --device cuda

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/home/jupyter/envhf/lib/python3.10/site-packages/optimum/exporters/onnx/model_configs.py:2680: UserWarning: Exporting model with image `height=64` which is less than minimal 320, setting `height` to 320.
  warnings.warn(
/home/jupyter/envhf/lib/python3.10/site-packages/optimum/exporters/onnx/model_configs.py:2686: UserWarning: Exporting model with image `width=64` which is less than minimal 320, setting `width` to 320.
  warnings.warn(
/home/jupyter/envhf/lib/python3.10/site-packages/transformers/models/rt_detr/modeling_rt_detr_resnet.py:107: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of 

pre-train

In [3]:
from transformers import AutoModelForObjectDetection

# Load your model
model_name = "/home/jupyter/rtdetr_101"
redetr_model = AutoModelForObjectDetection.from_pretrained(model_name)
# redetr_model.save_pretrained("quantized_model/redetr", save_config=True, safe_serialization=False)
# could make use of PR https://github.com/huggingface/optimum/pull/1930 to run the following

In [5]:
!optimum-cli export onnx -m /home/jupyter/rtdetr_101 --task object-detection --framework 'pt' hf_pt_onnx  --device cuda

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/home/jupyter/envhf/lib/python3.10/site-packages/optimum/exporters/onnx/model_configs.py:2680: UserWarning: Exporting model with image `height=64` which is less than minimal 320, setting `height` to 320.
  warnings.warn(
/home/jupyter/envhf/lib/python3.10/site-packages/optimum/exporters/onnx/model_configs.py:2686: UserWarning: Exporting model with image `width=64` which is less than minimal 320, setting `width` to 320.
  warnings.warn(
/home/jupyter/envhf/lib/python3.10/site-packages/transformers/models/rt_detr/modeling_rt_detr_resnet.py:107: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of 